# Tutorial: clasificación con k-Nearest Neighbors (kNN) en Python / scikit-learn
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Dataset:** Breast Cancer Prediction (Kaggle) · **Entorno:** Google Colab (también funciona en Jupyter)

> 💡 En Colab todas las librerías ya vienen instaladas. Ejecuta las celdas en orden con `Shift + Enter`.

En este tutorial implementaremos paso a paso la técnica **kNN** para clasificar tumores de mama como **malignos** o **benignos**, recorriendo todas las etapas de un *pipeline* de Machine Learning.

**Contenido**
1. Fundamentos teóricos de kNN
2. Abstracción matemática
3. Pipeline de Machine Learning
   - 3.1 Importación de librerías
   - 3.2 Recolección de datos
   - 3.3 Descripción y análisis del dataset (EDA)
   - 3.4 Preprocesamiento y limpieza
   - 3.5 Scatter plot
   - 3.6 División entrenamiento / prueba
   - 3.7 Escalado de características
   - 3.8 Selección del hiperparámetro k
   - 3.9 Entrenamiento
   - 3.10 Predicción
   - 3.11 Evaluación (métricas)
   - 3.12 Frontera de decisión
   - 3.13 Predicción de un caso nuevo
4. Conclusiones

**Referencias:** [Dataset en Kaggle](https://www.kaggle.com/datasets/merishnasuwal/breast-cancer-prediction-dataset) · [Tutorial de referencia en Medium](https://medium.com/@marciojmo/breast-cancer-classifier-using-the-k-nearest-neighbors-knn-algorithm-871fc4557246)

> ⚠️ Este notebook es **educativo**. Un modelo de este tipo no sustituye un diagnóstico médico.

---
# 1. Fundamentos teóricos de kNN

**k-Nearest Neighbors (k vecinos más cercanos)** es un algoritmo de aprendizaje **supervisado** que sirve para clasificación y regresión. Su idea es muy intuitiva:

> *"Dime con quién andas y te diré quién eres."* Un punto nuevo se clasifica según la clase predominante entre sus **k** vecinos más cercanos.

**Características clave**

| Característica | Descripción |
|---|---|
| **Aprendizaje perezoso (*lazy*)** | No construye un modelo explícito; solo **memoriza** los datos de entrenamiento. El trabajo ocurre al predecir. |
| **No paramétrico** | No asume una distribución de los datos ni una forma funcional. |
| **Basado en instancias** | La predicción depende directamente de ejemplos almacenados. |
| **Sensible a la escala** | Como usa distancias, las variables deben estar en escalas comparables. |

**Funcionamiento paso a paso**
1. Elegir el número de vecinos **k** y una **métrica de distancia**.
2. Para un punto nuevo, calcular su distancia a **todos** los puntos de entrenamiento.
3. Seleccionar los **k** puntos más cercanos.
4. Asignar la clase **mayoritaria** entre esos k vecinos (votación).

**Ventajas:** simple, sin fase de entrenamiento, funciona con fronteras de decisión no lineales.
**Desventajas:** predicción lenta con muchos datos, sensible a variables irrelevantes, a la escala y a la *maldición de la dimensionalidad*.

**Efecto de k:** un k pequeño (p. ej. 1) genera fronteras muy irregulares y **sobreajuste** (alta varianza); un k muy grande suaviza demasiado y provoca **subajuste** (alto sesgo).

---
# 2. Abstracción matemática

Sea un conjunto de entrenamiento $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^{n}$, donde $\mathbf{x}_i \in \mathbb{R}^d$ es el vector de características y $y_i \in \{0,1\}$ la clase (0 = maligno, 1 = benigno).

### 2.1 Distancia
Dado un punto nuevo $\mathbf{x}_q$, se mide su cercanía con una distancia. La más usada es la **euclidiana**:

$$d(\mathbf{x}_q, \mathbf{x}_i) = \sqrt{\sum_{j=1}^{d}\left(x_{q,j} - x_{i,j}\right)^2}$$

Casos particulares de la **distancia de Minkowski** de orden $p$:

$$d_p(\mathbf{x}_q, \mathbf{x}_i) = \left(\sum_{j=1}^{d}\left|x_{q,j} - x_{i,j}\right|^p\right)^{1/p}$$

- $p = 1$ → distancia de **Manhattan**
- $p = 2$ → distancia **Euclidiana** (valor por defecto en scikit-learn)

### 2.2 Conjunto de vecinos
Sea $N_k(\mathbf{x}_q)$ el conjunto de los $k$ puntos de $\mathcal{D}$ con menor distancia a $\mathbf{x}_q$.

### 2.3 Regla de decisión (voto mayoritario)

$$\hat{y}(\mathbf{x}_q) = \underset{c \in \{0,1\}}{\arg\max} \sum_{(\mathbf{x}_i, y_i) \in N_k(\mathbf{x}_q)} \mathbb{1}\left(y_i = c\right)$$

donde $\mathbb{1}(\cdot)$ vale 1 si la condición se cumple y 0 en caso contrario.

### 2.4 Variante con pesos por distancia
Los vecinos más cercanos pueden influir más:

$$\hat{y}(\mathbf{x}_q) = \underset{c}{\arg\max} \sum_{i \in N_k(\mathbf{x}_q)} w_i \, \mathbb{1}(y_i = c), \qquad w_i = \frac{1}{d(\mathbf{x}_q, \mathbf{x}_i)}$$

### 2.5 Probabilidad estimada
$$P(y = c \mid \mathbf{x}_q) \approx \frac{1}{k}\sum_{i \in N_k(\mathbf{x}_q)} \mathbb{1}(y_i = c)$$

### 2.6 Escalado (estandarización)
Para que ninguna variable domine la distancia por su magnitud:

$$z_j = \frac{x_j - \mu_j}{\sigma_j}$$

### 2.7 Complejidad
- Entrenamiento: $O(1)$ (solo se almacenan los datos).
- Predicción (fuerza bruta): $O(n \cdot d)$ por consulta.

---
# 3. Pipeline de Machine Learning

Seguiremos el flujo clásico:

`Recolección → Descripción/EDA → Preprocesamiento → Visualización → Partición → Escalado → Selección de k → Entrenamiento → Predicción → Evaluación`

## 3.1 Importación de librerías

In [ ]:
# --- Manipulación de datos ---
import numpy as np                      # Cálculo numérico
import pandas as pd                     # Tablas (DataFrames)

# --- Visualización ---
import matplotlib.pyplot as plt         # Gráficos base
import seaborn as sns                   # Gráficos estadísticos

# --- scikit-learn: preparación de datos ---
from sklearn.model_selection import train_test_split, cross_val_score  # Partición y validación cruzada
from sklearn.preprocessing import StandardScaler                       # Estandarización

# --- scikit-learn: modelo y métricas ---
from sklearn.neighbors import KNeighborsClassifier                     # Algoritmo kNN
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve)
from sklearn.inspection import DecisionBoundaryDisplay                 # Para graficar la frontera de decisión

# Estilo de las gráficas y semilla para reproducibilidad
sns.set_theme(style="whitegrid")
SEMILLA = 42
np.random.seed(SEMILLA)

## 3.2 Recolección de datos

El dataset de Kaggle (*Breast Cancer Prediction Dataset*) contiene **569 registros** con 5 variables numéricas medidas sobre imágenes de células y el diagnóstico:

| Variable | Descripción |
|---|---|
| `mean_radius` | Radio medio del núcleo celular |
| `mean_texture` | Textura media (desviación estándar de los niveles de gris) |
| `mean_perimeter` | Perímetro medio |
| `mean_area` | Área media |
| `mean_smoothness` | Suavidad media (variación local de la longitud del radio) |
| `diagnosis` | **0 = maligno**, **1 = benigno** |

**Cómo cargarlo**
- **Opción A (recomendada):** descarga `Breast_cancer_data.csv` desde Kaggle. En **Google Colab**, al ejecutar la celda siguiente se abrirá un selector para subirlo (también puedes arrastrarlo al panel de archivos 📁 antes de ejecutar).
- **Opción B (automática):** si el archivo no se encuentra, el notebook usa el dataset incluido en scikit-learn, que contiene **exactamente las mismas mediciones** (Breast Cancer Wisconsin), seleccionando las mismas 5 variables.

In [ ]:
import os, sys

# Ruta/nombre esperado del CSV descargado de Kaggle
RUTA_CSV = "Breast_cancer_data.csv"

# ¿Estamos corriendo en Google Colab?
EN_COLAB = "google.colab" in sys.modules

# --- Google Colab: si el CSV no está en el entorno, abrimos el selector de archivos ---
if EN_COLAB and not os.path.exists(RUTA_CSV):
    from google.colab import files
    print("📂 Selecciona el archivo CSV de Kaggle (o cancela para usar el dataset de respaldo)...")
    subido = files.upload()                  # Abre el cuadro para subir archivos desde tu computadora
    if subido:                               # Si se subió algo, usamos ese archivo
        RUTA_CSV = list(subido.keys())[0]

if os.path.exists(RUTA_CSV):
    # Opción A: cargar el CSV de Kaggle
    df = pd.read_csv(RUTA_CSV)
    print(f"✅ Dataset cargado desde '{RUTA_CSV}'")
else:
    # Opción B (respaldo): mismo dataset incluido en scikit-learn
    from sklearn.datasets import load_breast_cancer
    datos = load_breast_cancer(as_frame=True)
    columnas = ["mean radius", "mean texture", "mean perimeter", "mean area", "mean smoothness"]
    df = datos.frame[columnas + ["target"]].rename(columns={"target": "diagnosis"})
    print("ℹ️ CSV no encontrado: se usa el dataset equivalente de scikit-learn")

# Normalizamos los nombres de columnas (minúsculas, sin espacios)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

df.head()  # Primeras 5 filas

## 3.3 Descripción y análisis del dataset (EDA)

Antes de modelar hay que **entender los datos**: tamaño, tipos, estadísticos, balance de clases y relaciones entre variables.

In [ ]:
# Dimensiones: (filas, columnas)
print("Forma del dataset:", df.shape)

# Tipos de datos y valores no nulos
print("\n--- Información general ---")
df.info()

In [ ]:
# Estadísticos descriptivos de cada variable (media, desviación, mínimos, cuartiles, máximos)
df.describe().T

In [ ]:
# Balance de clases: ¿cuántos tumores malignos (0) y benignos (1) hay?
conteo = df["diagnosis"].value_counts().sort_index()
print(conteo)
print("\nProporción:\n", (conteo / len(df)).round(3))

# Gráfico de barras del balance de clases
plt.figure(figsize=(5, 4))
sns.countplot(x="diagnosis", data=df, hue="diagnosis", palette=["#d62728", "#2ca02c"], legend=False)
plt.xticks([0, 1], ["Maligno (0)", "Benigno (1)"])
plt.title("Distribución de la variable objetivo")
plt.xlabel("Diagnóstico"); plt.ylabel("Cantidad")
plt.show()

In [ ]:
# Histogramas de cada variable separados por clase
caracteristicas = [c for c in df.columns if c != "diagnosis"]

fig, ejes = plt.subplots(2, 3, figsize=(15, 8))
for eje, col in zip(ejes.ravel(), caracteristicas):
    sns.histplot(data=df, x=col, hue="diagnosis", kde=True, ax=eje,
                 palette=["#d62728", "#2ca02c"], element="step")
    eje.set_title(col)
ejes.ravel()[-1].axis("off")   # Ocultamos el eje sobrante (hay 5 variables y 6 espacios)
plt.tight_layout(); plt.show()

In [ ]:
# Matriz de correlación: ¿qué variables están relacionadas entre sí y con el diagnóstico?
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Matriz de correlación")
plt.show()

**Observaciones:**
- Las variables de tamaño (`radius`, `perimeter`, `area`) están **fuertemente correlacionadas** entre sí (miden aspectos similares).
- Los tumores **malignos** (0) tienden a tener radio, perímetro y área **mayores**.
- Las escalas son muy distintas (`mean_area` ronda cientos/miles, `mean_smoothness` ronda 0.1): **el escalado será indispensable** para kNN.

## 3.4 Preprocesamiento y limpieza

In [ ]:
# Valores nulos por columna
print("Valores nulos:\n", df.isnull().sum())

# Filas duplicadas
print("\nFilas duplicadas:", df.duplicated().sum())

# Eliminamos duplicados (si los hubiera) y filas con nulos
df = df.drop_duplicates().dropna().reset_index(drop=True)

# Separamos la matriz de características X y el vector objetivo y
X = df.drop(columns="diagnosis")   # Variables predictoras
y = df["diagnosis"]                # Variable objetivo (0 = maligno, 1 = benigno)

print("\nX:", X.shape, "| y:", y.shape)

## 3.5 Scatter plot

Visualizamos la separación entre clases usando pares de variables. kNN funciona bien cuando **puntos de la misma clase están cerca entre sí**.

In [ ]:
# Scatter plot principal: radio vs textura, coloreado por diagnóstico
etiquetas = df["diagnosis"].map({0: "Maligno (0)", 1: "Benigno (1)"})   # Nombres legibles para la leyenda
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="mean_radius", y="mean_texture", hue=etiquetas,
                palette={"Maligno (0)": "#d62728", "Benigno (1)": "#2ca02c"},
                alpha=0.7, edgecolor="k")
plt.title("Scatter plot: radio medio vs textura media")
plt.legend(title="Diagnóstico")
plt.show()

In [ ]:
# Scatter plot de todos los pares de variables (pairplot)
sns.pairplot(df, hue="diagnosis", palette=["#d62728", "#2ca02c"], corner=True, plot_kws={"alpha": 0.6, "s": 20})
plt.suptitle("Pairplot de las variables", y=1.02)
plt.show()

**Lectura:** en `mean_radius` vs `mean_texture` se aprecia una separación bastante clara, con una zona de solapamiento. En esa zona es donde kNN puede equivocarse.

## 3.6 División en entrenamiento y prueba

Separamos los datos para evaluar el modelo con ejemplos que **no vio** durante el entrenamiento. Usamos `stratify=y` para conservar la proporción de clases en ambos conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,       # 25 % para prueba, 75 % para entrenamiento
    random_state=SEMILLA, # Reproducibilidad
    stratify=y            # Mantiene la proporción de clases
)

print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("\nProporción de clases en entrenamiento:\n", y_train.value_counts(normalize=True).round(3))
print("\nProporción de clases en prueba:\n", y_test.value_counts(normalize=True).round(3))

## 3.7 Escalado de características

kNN calcula distancias, así que una variable con valores grandes (`mean_area`) dominaría a una pequeña (`mean_smoothness`). Aplicamos **estandarización** ($z = (x-\mu)/\sigma$).

> 🔑 **Regla de oro:** el escalador se ajusta (`fit`) **solo con el conjunto de entrenamiento** y luego se aplica (`transform`) a entrenamiento y prueba. Así evitamos **fuga de datos** (*data leakage*).

In [ ]:
escalador = StandardScaler()                           # Crea el estandarizador

X_train_esc = escalador.fit_transform(X_train)         # Aprende media/desv. en train y transforma
X_test_esc  = escalador.transform(X_test)              # Aplica los MISMOS parámetros a test

# Verificación: tras el escalado, cada columna de train tiene media ≈ 0 y desviación ≈ 1
print("Media (train):", X_train_esc.mean(axis=0).round(3))
print("Desv. (train):", X_train_esc.std(axis=0).round(3))

## 3.8 Selección del hiperparámetro k

No existe un k universal. Lo elegimos con **validación cruzada de 5 pliegues** sobre el conjunto de entrenamiento, probando varios valores y comparando la exactitud media.

*(Probamos valores impares para evitar empates en la votación con 2 clases.)*

In [ ]:
valores_k = range(1, 31, 2)       # k = 1, 3, 5, ..., 29
exactitud_media = []              # Exactitud promedio por cada k

for k in valores_k:
    modelo = KNeighborsClassifier(n_neighbors=k)                       # kNN con k vecinos
    puntajes = cross_val_score(modelo, X_train_esc, y_train, cv=5,     # Validación cruzada 5-fold
                               scoring="accuracy")
    exactitud_media.append(puntajes.mean())                            # Guardamos el promedio

# Mejor k = el de mayor exactitud media
mejor_k = list(valores_k)[int(np.argmax(exactitud_media))]
print(f"Mejor k encontrado: {mejor_k}  (exactitud CV = {max(exactitud_media):.4f})")

# Gráfica exactitud vs k
plt.figure(figsize=(8, 4))
plt.plot(list(valores_k), exactitud_media, marker="o")
plt.axvline(mejor_k, color="red", linestyle="--", label=f"Mejor k = {mejor_k}")
plt.title("Selección de k mediante validación cruzada")
plt.xlabel("k (número de vecinos)"); plt.ylabel("Exactitud media (CV)")
plt.legend(); plt.show()

## 3.9 Entrenamiento del modelo

In [ ]:
# Creamos el clasificador kNN con el mejor k
knn = KNeighborsClassifier(
    n_neighbors=mejor_k,   # Número de vecinos
    weights="uniform",     # Todos los vecinos pesan igual ('distance' ponderaría por cercanía)
    metric="minkowski",    # Métrica de Minkowski...
    p=2                    # ...con p=2 equivale a distancia euclidiana
)

# 'fit' en kNN solo almacena los datos de entrenamiento (aprendizaje perezoso)
knn.fit(X_train_esc, y_train)
print("Modelo entrenado:", knn)

## 3.10 Predicción

In [ ]:
# Clases predichas para el conjunto de prueba
y_pred = knn.predict(X_test_esc)

# Probabilidad estimada de cada clase (fracción de vecinos de cada clase)
y_prob = knn.predict_proba(X_test_esc)[:, 1]   # Probabilidad de ser benigno (clase 1)

# Comparación de las primeras 10 predicciones con los valores reales
comparacion = pd.DataFrame({"Real": y_test.values[:10],
                            "Predicho": y_pred[:10],
                            "P(benigno)": y_prob[:10].round(2)})
comparacion

## 3.11 Evaluación del modelo (métricas)

En diagnóstico médico no basta con la exactitud: importa **qué tipo de error** se comete. Definiendo "maligno" como la clase de interés:

- **Falso negativo:** un tumor maligno clasificado como benigno → *error más grave*.
- **Falso positivo:** un tumor benigno clasificado como maligno → genera estudios adicionales.

**Métricas:**

$$\text{Exactitud}=\frac{TP+TN}{TP+TN+FP+FN}\quad
\text{Precisión}=\frac{TP}{TP+FP}\quad
\text{Recall}=\frac{TP}{TP+FN}\quad
F_1=2\cdot\frac{\text{Precisión}\cdot\text{Recall}}{\text{Precisión}+\text{Recall}}$$

In [ ]:
# --- Exactitud global ---
exactitud = accuracy_score(y_test, y_pred)
print(f"Exactitud en prueba: {exactitud:.4f}")

# --- Reporte de clasificación (precisión, recall, F1 por clase) ---
print("\nReporte de clasificación:\n")
print(classification_report(y_test, y_pred, target_names=["Maligno (0)", "Benigno (1)"]))

In [ ]:
# --- Matriz de confusión ---
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Maligno", "Benigno"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Matriz de confusión")
plt.show()

# Desglose: con clases [0, 1], ravel() devuelve TN, FP, FN, TP
tn, fp, fn, tp = cm.ravel()
print(f"Malignos detectados correctamente (TN): {tn}")
print(f"Benignos clasificados como malignos (FP): {fp}")
print(f"Malignos clasificados como benignos (FN, error grave): {fn}")
print(f"Benignos detectados correctamente (TP): {tp}")

In [ ]:
# --- Curva ROC y AUC ---
auc = roc_auc_score(y_test, y_prob)                 # Área bajo la curva
fpr, tpr, _ = roc_curve(y_test, y_prob)             # Puntos de la curva

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"kNN (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Azar")       # Línea de referencia
plt.xlabel("Tasa de falsos positivos"); plt.ylabel("Tasa de verdaderos positivos")
plt.title("Curva ROC"); plt.legend(); plt.show()

## 3.12 Frontera de decisión (visualización en 2D)

Para **ver** cómo decide kNN, entrenamos un modelo auxiliar usando solo 2 variables (`mean_radius` y `mean_texture`) y dibujamos las regiones de cada clase.

In [ ]:
# Seleccionamos 2 variables para poder graficar en el plano
cols_2d = ["mean_radius", "mean_texture"]

# Escalamos solo estas dos columnas (ajustando con train)
esc_2d = StandardScaler().fit(X_train[cols_2d])
X_tr_2d = esc_2d.transform(X_train[cols_2d])
X_te_2d = esc_2d.transform(X_test[cols_2d])

# Entrenamos un kNN auxiliar con las 2 variables
knn_2d = KNeighborsClassifier(n_neighbors=mejor_k).fit(X_tr_2d, y_train)
print(f"Exactitud con solo 2 variables: {knn_2d.score(X_te_2d, y_test):.4f}")

# Dibujamos las regiones de decisión y los puntos de prueba
fig, eje = plt.subplots(figsize=(7, 5))
DecisionBoundaryDisplay.from_estimator(knn_2d, X_tr_2d, cmap="RdYlGn", alpha=0.4,
                                       response_method="predict", ax=eje,
                                       xlabel="mean_radius (escalado)", ylabel="mean_texture (escalado)")
eje.scatter(X_te_2d[:, 0], X_te_2d[:, 1], c=y_test, cmap="RdYlGn", edgecolor="k")
eje.set_title(f"Frontera de decisión kNN (k = {mejor_k})")
plt.show()

## 3.13 Predicción de un caso nuevo

In [ ]:
# Paciente hipotético (mismas 5 variables y mismo orden que X)
nuevo = pd.DataFrame([[14.0, 20.0, 90.0, 600.0, 0.10]], columns=X.columns)

nuevo_esc = escalador.transform(nuevo)             # Se escala con el escalador ya ajustado
clase = knn.predict(nuevo_esc)[0]                  # Clase predicha
probas = knn.predict_proba(nuevo_esc)[0]           # [P(maligno), P(benigno)]

print("Diagnóstico predicho:", "Benigno" if clase == 1 else "Maligno")
print(f"P(maligno) = {probas[0]:.2f} | P(benigno) = {probas[1]:.2f}")

# Vecinos más cercanos que votaron por este caso
distancias, indices = knn.kneighbors(nuevo_esc)
print("\nDistancias a los vecinos:", distancias.round(2)[0])
print("Clases de los vecinos:", y_train.iloc[indices[0]].values)

---
# 4. Conclusiones

- kNN es un algoritmo **simple e interpretable**: clasifica según la mayoría de los $k$ vecinos más cercanos.
- Su desempeño depende de tres decisiones: la **métrica de distancia**, el **valor de k** y el **escalado** de las variables.
- Elegimos $k$ con **validación cruzada** y evaluamos con datos no vistos, usando exactitud, precisión, recall, F1, matriz de confusión y AUC.
- En un contexto médico se debe vigilar especialmente el **recall de la clase maligna** (minimizar falsos negativos).

**Limitaciones:** costo computacional alto con muchos datos, sensibilidad a variables irrelevantes y a la dimensionalidad.

**Para seguir explorando:**
- Probar `weights="distance"` y `p=1` (Manhattan).
- Usar las 30 variables del dataset completo de scikit-learn.
- Comparar con otros clasificadores (Perceptrón, SVM, regresión logística).
- Encapsular el flujo en un `Pipeline` de scikit-learn con `GridSearchCV`.